[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eirasf/GCED-AA3/blob/main/en/lab6/lab6.ipynb)

# Lab6: Reinforcement learning - Temporal Difference methods

In this lab we will continue exploring tabular reinforcement learning methods that do not need to have a model. In particular, we will study the **Temporal Difference** methods.

To explore these methods we will use [Gym](https://www.gymlibrary.dev/) again with a GridWorld environment, that is, a board with cells through which the agent moves, just as we did in Lab4. In particular, we are going to use the environment `MiniGrid-DistShift1-v0`. In this environment the agent must reach the goal, but this time avoiding falling into the lava, which makes the episode end with reward 0.

Let's load the environment and explore it briefly.

In [ ]:
# If the packages are not installed, you have to run these lines:
#!pip install gymnasium[classic-control]
#!pip install minigrid
import gymnasium as gym
import minigrid
import numpy as np
env = gym.make('MiniGrid-DistShift1-v0', render_mode='rgb_array')

import matplotlib.pyplot as plt

# TODO - Show the environment to see what the board looks like. To do so, retrieve the show_environment function from lab4 and run it
...

In [ ]:
# TODO - Record the dimensions of the board to be able to use them later
NUM_COLUMNS = ...
NUM_ROWS = ...
NUM_ORIENTATIONS = 4

# The actions are the same as in lab 4. We will again keep only LEFT, RIGHT and FORWARD
actions = env.actions
# We select only the three indicated actions
USEFUL_ACTIONS = [actions.left, actions.right, actions.forward]
NUM_ACTIONS = len(USEFUL_ACTIONS)

Check the effect of falling into the lava. The episode should end and the agent should receive a reward of 0.

In [ ]:
#TODO - Perform two actions so that the agent falls into the lava and verify that a reward of 0 is received
obs, reward, terminated, truncated, info = ...
obs, reward, terminated, truncated, info = ...

#CHECKS
assert(reward==0)
assert(terminated)
_ = env.reset()

# Obtaining optimal policies

## Temporal Difference methods

The methods based on **Temporal Difference**, like the Montecarlo methods, lack information about how the model works (they do not know $p(s',r | s,a)$). That is why they must also interact with the environment to obtain samples with which to later make estimations.

Unlike the Montecarlo methods, the Temporal Difference methods do not wait to finish an episode to update the values of the states traversed. Instead, the Temporal Difference methods take advantage of the idea that the value $v(S)$ of a state $S$ is related to the value of the states $S'$ that can be reached from $S$. In other words, the states that can lead to a bad state will also be bad (and vice versa). This idea is encoded in the **Bellman equations**:

$$v_\pi(s)=\sum_{a}\pi(a|s)\sum_{s',r} p(s',r | s,a)\left[r + \gamma v_\pi(s')\right]$$

The same idea can be applied when estimating the value of executing action $a$ being in state $s$. The actions $a$ that lead to states $s'$ whose actions have high values will, in turn, have high values (and vice versa).

$$q_\pi(s,a)=\sum_{s',r} p(s',r | s,a)\left[r + \gamma \sum_{a'}\pi(a'|s')q_\pi(s',a')\right]$$

Applying this idea will allow us to do *bootstrapping* in our calculations: when calculating the value of a state we will be able to take advantage of the approximation we have for adjacent states.

### SARSA
The first method we will try is SARSA. This algorithm is named after the variables involved in each update step of the estimation of $Q(S,A)$. The variables are $S_t,A_t,R_{t+1},S_{t+1},A_{t+1}$ and the update uses this formula:

$$Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha [R_{t+1} + \gamma Q(S_{t+1},A_{t+1}) - Q(S_t,A_t)]$$

The algorithm will execute repeated episodes (potentially infinite; it is a control algorithm and the agent could keep using it during its entire existence) and, for each step of the episode, it will make the update according to the formula indicated above. This requires that the next action to take ($A_{t+1}$) has been decided before updating the value of the previous action ($A_t$).

![Sarsa](./img/sarsa.png)

To facilitate the implementation, let's first write a couple of auxiliary functions that allow us to represent states and sample actions from $Q$.

In [ ]:
GAMMA = 0.9 # For this problem we are going to use a discount factor of 0.9, that is, future rewards will be discounted 10% for each step that is necessary to obtain them.

env.max_steps = 5000 # We set the maximum number of actions per episode to 5000, to allow long episodes

# AUXILIARY FUNCTIONS
#TODO - Retrieve the get_state function from lab 4. This function encodes the state of the environment in a list of three elements: column, row, orientation
def get_state(env:gym.Env) -> State:
    ...

# CHECK
env.reset()
current_state = get_state(env)
assert current_state.x == 0, f'The state right after resetting must indicate x=0 and yours indicates {current_state.x}'
assert current_state.y == 0, f'The state right after resetting must indicate y=0 and yours indicates {current_state.y}'
assert current_state.dir == 0, f'The state right after resetting must indicate dir=0 and yours indicates {current_state.dir}'

# TODO - Write a function that, given the q values of the different actions in a specific state, returns an action making an epsilon-greedy selection from the q_values.
def get_epsilon_greedy_action(q_values:np.ndarray, epsilon:float):
    '''
    Selects an action based on the provided q_values. A fraction of the times (indicated by epsilon) it will return a random action

    Arguments:
    q_values -- List with the q values of the actions among which to select.
    epsilon -- Fraction of the times that a random action will be returned
    '''
    if np.random.random()<epsilon:
        # TODO - Return a random action
        return ...
    # TODO - Return the index of the biggest q value
    return ...


# CHECK
assert(get_epsilon_greedy_action([1,2,4],0)==2)

For algorithms derived from SARSA we will do _exploring starts_, that is, they will start each time in a random state. This facilitates exploration and, therefore, the training, since it facilitates that episodes occur that the agent is able to solve even with a random policy.

We will have to create functions that allow us to carry this out.

In [ ]:
# TODO - Write a function that generates the encoding for a random valid state. It will return a list with three values that indicate column, row and orientation
def get_random_state() -> State:
    ...

# TODO - Write a function that sets the environment so that it matches what is indicated by a given state
def set_state(env:gym.Env, state: State):
    ...

# CHECK
env.reset()
show_environment(env) # It must show the agent in cell 0,0
set_state(env, State(x=2, y=3, dir=0))
show_environment(env) # It must show the agent in cell 2,3

We already have what is necessary to implement SARSA. We are going to establish a fixed number of iterations and make it return the $Q$ values when it finishes. In addition, we will include messages at the end of each episode that indicate:
 - The number of the episode that has finished
 - The reward obtained
 - The number of actions necessary to complete the episode
 - The average reward obtained during the last 100 episodes (if at least 100 have been executed)

In [ ]:
def sarsa(num_episodes:int = 500, ALPHA:float = 0.1, EPSILON:float = 0.25) -> np.ndarray:
    # TODO - Initialize the Q values to zero with the appropriate shape
    q_values = ...

    # In last100returns we will store the returns of the last 100 episodes
    last100returns = []
    # TODO - Do as many episodes as num_episodes indicates
        # TODO - Return the environment to its initial state
        # TODO - Initialize S
        # TODO - Choose A from S using policy derived from Q (e.g. epsilon-greedy)

        G = 0
        num_steps = 0
        # TODO - Loop for each step of episode (repeat until the episode ends)
            num_steps += 1
            # TODO - Take action A, observe R, S'
            # TODO - Accumulate the reward R to returns to be able to print the return of the episode
            # TODO - Choose A' from S' using policy derived from Q (e.g. epsilon-greedy)
            # TODO - Q(S,A) <- Q(S,A) + ALPHA * [R + GAMMA * Q(S',A') - Q(S,A)]
            # TODO - S <- S'; A <- A'

        # After finishing the episode we store the return in the last 100...
        last100returns.append(G)
        avg_returns_str = ''
        if len(last100returns)==101: # ... and if there are already more than 100...
            last100returns.pop(0) # ...we remove the oldest one (to always have 100, not more)...
            #... and we prepare the message to show the mean of the returns
            avg_returns_str=f'(mean return of {np.mean(last100returns)} in the last 100 episodes)'
        # We show the message after each episode
        print(f'Finished episode {i} with return {G} in {num_steps} steps {avg_returns_str}')
    return q_values

# We train the agent using the default parameters of the function we have just defined
sarsa_q_values = sarsa()

The learning process has an important random component, so two consecutive executions can learn different $Q$ values from which different policies $\pi$ will also be derived. However, if you have implemented the algorithm well, you should have observed the following:
  1. Initially there are many episodes with return 0 (the agent ends up in the lava). The duration of these episodes is very variable. In addition, if we printed `q_values` after each of these episodes, we would see that $Q$ is not modified! Each step uses $R$ and $Q(S',A')$ to update $Q(S,A)$, but if both $R$ and $Q(S',A')$ are zero, the value of $Q(S,A)$ will not change.
  1. Over time, and by pure chance, the agent will finish some episode reaching the goal. This will give it a reward of 1, which will make $Q(S_t,A_t)$ update to a positive value.
  1. The next time the agent is in $S_t$, it will choose the action $A_t$ (unless the random action comes up because of the $\epsilon$), which will make it finish the episode with reward 1. **It will have learned a useful policy for the last cell!** But, in addition, the state $S_{t-1}$ that led to $S_t$ (using the action $A_{t-1}$) will see its $Q(S_{t-1},A_{t-1})$ updated to a positive value. **Now it will know a useful policy for the last two cells!**
  1. Through this process, the information regarding which policy is useful will propagate from the cells adjacent to the goal to their neighbors, and from these to their neighbors and so on until covering the whole board. It must be added that, also by pure chance, the agent can find other paths that lead to the goal and launch this same propagation process but following another path that leads to the goal.
  1. The effect of this is that, as the episodes go by, two things happen:
    - The number of episodes with positive return increases
    - The duration of the episodes with positive return decreases (the agent goes more time through "known path")
  1. This has an impact on the mean return of the last 100 episodes: it will start being tiny, but at some point it will rise very quickly.
  1. The return of the last 100 episodes will reach a maximum point around which it will oscillate towards the end.


Let's visualize the learned policy. We are going to show, for each cell, the orientation that the agent has when the preferred action (the one with the highest $Q$ value) is `FORWARD`.

>**Reminder**
>
> The orientations are encoded like this:
> - 0 $\rightarrow$ right
> - 1 $\rightarrow$ down
> - 2 $\rightarrow$ left
> - 3 $\rightarrow$ up

In [ ]:
# This function returns an arrow character that represents the orientation of the agent for which the q values recommend the forward action
def get_max_value_direction_arrow(q_values:np.ndarray) -> str:
    preferred_action_per_orientation = q_values.argmax(axis=1) # For each one of the four orientations, we calculate which action is preferred
    direction = np.argmax(preferred_action_per_orientation == 2) # Of the orientations that prefer FORWARD, we take one (the first)
    # Depending on the orientation, we return the appropriate arrow.
    if direction==0:
        return '⮕'
    if direction==1:
        return '⬇'
    if direction==2:
        return '⬅'
    if direction==3:
        return '⬆'
    return '?'

# Shows the policy on screen
def draw_policy(q_values:np.ndarray) -> None:
    # Goes through the rows of the board...
    for i in range(q_values.shape[1]):
        # ... composing a line per row...
        line = ''
        for j in range(q_values.shape[0]): # Goes through the columns
            if i==0 and j==6: # We paint the goal
                    line+='🟥'
            elif (i==0 or i==1)and j>1 and j<5: # We paint the lava
                    line+='🟧'
            else:
                line+=get_max_value_direction_arrow(q_values[j,i]) # We paint the appropriate arrow for this cell
        # ...which finally prints
        print(line)

# TODO - Show the policy that is derived from the values learned by SARSA
...

We have verified that our agent has to do many episodes that do not lead it to learn because they do not give it any reward, which causes it not to be able to update $Q(S,A)$ for any of the $S,A$ pairs it goes through.

**What do you think would happen if we gave a negative reward for falling into the lava? Would the speed at which the agent learns change?**

Try it by adding this after each step:
```python
    if completed and reward==0:
        reward = -1
```

Modify the `sarsa` function and run the following cell. **What do you observe?**

In [ ]:
sarsa_q_values_negative_lava = sarsa() # Sarsa must have been modified so that falling into the lava gives a reward of -1
draw_policy(sarsa_q_values_negative_lava)

### Q-learning
The Q-learning algorithm is one of the most popular within Reinforcement Learning. It is very similar to SARSA, but it has an important difference: it is *off-policy*. If SARSA made the updates of $Q(S_t,A_t)$ based on the action $A_{t+1}$ that it took, Q-learning is going to make those updates regardless of the action it finally takes. Therefore, it will be using one policy to explore but another to update. Specifically, Q-learning updates according to this formula:

$$Q(S_t,A_t) \leftarrow Q(S_t,A_t) + \alpha [R_{t+1} + \gamma \max_aQ(S_{t+1},a) - Q(S_t,A_t)]$$

Q-learning does not update based on the action that is going to be taken, but based on the best possible action. The complete algorithm is described below.

![Q-learning](./img/q-learning.png)

Let's implement Q-learning showing the same messages at each step that we showed with SARSA. Keep the negative rewards for when the agent falls into the lava.

In [ ]:
# TODO - Implement q_learning
#  Use the same code as for sarsa but changing the update
#  Remember to keep the negative rewards for when the agent falls into the lava and to show the messages
def q_learning(num_episodes:int = 500, ALPHA:float = 0.1, EPSILON:float = 0.25) -> np.ndarray:
    ...

qlearning_q_values = q_learning()
draw_policy(qlearning_q_values)

### SARSA vs Q-Learning comparison
Compare the results obtained by both algorithms. You should observe the following:
  - The policy obtained by SARSA is more conservative (it does not want to be near the lava), while the one of Q-learning is more optimistic (it does not mind being near the lava; it trusts its policy).
  - Therefore, the episodes of Q-learning are shorter than those of SARSA.
  - However, when following an $\epsilon$-greedy policy, being in a cell adjacent to the lava results in a reward of -1 (falling into the lava) a $\frac{\epsilon}{4}$ of the times, so the mean return of Q-learning is lower.

Complete the following cell to check it.

In [ ]:
def check_derived_policy(q_values:np.ndarray, epsilon:float) -> None:
    env.reset()
    # TODO - Write a loop that simulates a complete episode following a COMPLETELY GREEDY policy with respect to q_values
    # and show its duration and its return
    ...
    print(f'The greedy policy makes an episode of {counter} steps with a return of {G}')

    env.reset()
    # TODO - Write another loop that simulates 500 episodes following an epsilon-greedy policy with respect to q_values
    # and show its mean duration and its mean return
    ...
    print(f'The greedy policy makes episodes of {np.mean(durations)} steps on average with a return of {np.mean(returns)} on average')

print('SARSA')
check_derived_policy(sarsa_q_values_negative_lava, 0.25)
draw_policy(sarsa_q_values_negative_lava)
print('\nQ-LEARNING')
check_derived_policy(qlearning_q_values, 0.25)
draw_policy(qlearning_q_values)